   The Jacobian matrix in robotics is a mathematical tool that maps a robot's 
   joint velocities to the linear and angular velocities of its end-effector
   (like a gripper or tool). It is the foundation for robot motion control,
   trajectory planning, and calculating how forces translate from the joints to
   the environment.

THE CORE CONCEPT: MAPPING VELOCITIES
   Mathematically, the relationship is expressed as:
   ...
   

---

   The file is the actual "brain" of your staged controller. It implements what
   ... a FSM is. Instead of trying to calculate one massive, complex math
   path for the robot, it breaks the insertion task down into bite-sized,
   sequential phases.

   ... breakdown of how each function works together to drive the robot arm.

---
1. THE CORE LIFECYCLE FUNCTIONS
   These are the main functions you will call from your `main.c` benchmark loop.
   - `scripted_policy_init` (THE CONSTRUCTOR): You run this exactly once when
     your program boots. It sets up the memory. More importantly, it queries
     MuJoCo to find the integer `actuator_ids` for your 7 motors so it never
     has to do slow string lookups during the live simulation.
   - `scripted_policy_start` (THE TRIAL RESET): You run this at the start of
     every single insertion trial. It reads the robot's current resting physical
     pose and saves it as `home_qpos`. This is vital because all your "tweaked 
     by eye" offsets are added to this baseline. It then officially kicks off
     the `SP_APPROACH` state.
   - `scripted_policy_update` (THE HEARTBEAT): This is the most important 
     function in the file. You call this inside your `for` loop every single
     time you call `mj_step`.
      1. It checks what state it is current in and how long that should take.
      2. It calculates $t$, a percentage from 0.0 to 1.0 representing how much
         time has passed in the current state.
      3. It feeds that $t$ into your `traj_lerp_array` (from...), calculating
         where exactly the 7 motors should be at this exact millisecond.
      4. It sends those calculated commands to MuJoCo.
      5. If time is up (t >= 1.0), it triggers the transition to the next state.


2. THE STATE MACHINE ENGINE
   These functions handle the transitions between the different phases of 
   movement.
   - `enter_state` (The Gear Shifter):   
     

---

WHAT IS `command_qpos`?
   `command_qpos` IS THE LIVE, INSTANTANEOUS INSTRUCTION SEN TO THE MOTORS AT
   THIS EXACT MILLISECOND.

   Think of `start_qpos` and `target_qpos` as the two endpoints of a straight 
   line. 